# gbm_os — Usage Guide

End-to-end walkthrough of the `gbm_os` cohort-access package: loading the manifest, exploring the data, selecting a study cohort, and feeding it to PyTorch / MONAI / torchio.

**Prerequisites**
- `output/master_manifest.csv` built by `manifest build`
- Raw datasets mounted at `/mnt/disk1/datasets/`
- `source activate.sh` (or venv active) so `gbm_os` and `gbm_manifest` are importable
- **Framework backends** require PyTorch — install torch with your CUDA build first:
  `pip install torch --index-url https://download.pytorch.org/whl/cu121`  
  then: `pip install gbm-manifest[all]`  (or individually: `[monai]`, `[torchio]`)


## 1. Load the Cohort

In [1]:
from pathlib import Path
from gbm_os import Cohort, CohortConfig

config = CohortConfig(
    data_roots={
        "brats2020": "/mnt/disk1/datasets/BraTS-2020",
        "rhuh_gbm":  "/mnt/disk1/datasets/RHUH-GBM",
        "upenn_gbm": "/mnt/disk1/datasets/UPENN-GBM",
        "ucsf_pdgm": "/mnt/disk1/datasets/UCSF-PDGM",
    },
    external={"upenn_gbm"},                              # UPENN = held-out test set
    priority=["brats2020", "rhuh_gbm", "ucsf_pdgm"],   # dedup winner order
)

MANIFEST = Path("../output/master_manifest.csv")

cohort = Cohort.from_manifest(MANIFEST, data_roots=config.data_roots, config=config)
print("Manifest loaded.")

Manifest loaded.


## 2. Explore with cohort_summary

Before selecting anything, get a full picture of what is in the manifest: demographics, survival, modality completeness, and clinical variable distributions.

In [2]:
from gbm_os import cohort_summary

full_view = cohort.select()
s = cohort_summary(full_view)
s.print()

════════════════════════════════════════════════════════════════════════
  COHORT SUMMARY   1540 patients · 1661 sessions · 4 datasets
════════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────
  Demographics & survival  (patient-level)
────────────────────────────────────────────────────────────────────────
           n_pat  n_ses  n_long  age_μ  age_med  age_σ  age_miss%   os_μ  os_med   os_σ  os_miss%  event%
dataset                                                                                                  
brats2020    369    369       0   61.2     61.5   11.9       36.0  445.5   369.0  355.2      36.0    99.6
rhuh_gbm      40    120      40   63.0     64.0    9.2        0.0  437.4   364.0  272.4       0.0    77.5
ucsf_pdgm    501    501       0   56.9     59.0   15.0        0.0  575.3   421.0  517.1       0.2    50.1
upenn_gbm    630    671      41   62.7     63.4   12.5        0.0  488.1

In [3]:
# Access individual tables as DataFrames
s.clinical       # patient-level demographics & survival

,n_patients,n_sessions,n_longitudinal,age_mean,age_median,age_std,age_pct_missing,os_days_mean,os_days_median,os_days_std,os_days_pct_missing,event_rate_pct
dataset,,,,,,,,,,,,
brats2020,369,369,0,61.2,61.5,11.9,36.0,445.5,369.0,355.2,36.0,99.6
rhuh_gbm,40,120,40,63.0,64.0,9.2,0.0,437.4,364.0,272.4,0.0,77.5
ucsf_pdgm,501,501,0,56.9,59.0,15.0,0.0,575.3,421.0,517.1,0.2,50.1
upenn_gbm,630,671,41,62.7,63.4,12.5,0.0,488.1,370.0,515.6,4.3,97.3
TOTAL,1540,1661,81,60.4,61.4,13.5,8.6,510.9,383.0,489.1,10.5,80.2


In [4]:
s.imaging        # session-level modality completeness

,n_sessions,pct_t1,pct_t1ce,pct_t2,pct_flair,pct_seg,pct_complete
dataset,,,,,,,
brats2020,369,100.0,100.0,100.0,100.0,99.7,100.0
rhuh_gbm,120,100.0,100.0,100.0,100.0,100.0,100.0
ucsf_pdgm,501,100.0,100.0,100.0,100.0,100.0,100.0
upenn_gbm,671,79.1,77.3,77.9,77.3,74.1,35.9
TOTAL,1661,91.6,90.8,91.1,90.8,89.5,74.1


In [5]:
s.distributions["eor"]    # EOR distribution per dataset

,brats2020,rhuh_gbm,ucsf_pdgm,upenn_gbm,TOTAL
eor,,,,,
missing,133 (36.0%),0 (0.0%),0 (0.0%),0 (0.0%),133 (8.6%)
GTR,119 (32.2%),27 (67.5%),248 (49.5%),326 (51.7%),720 (46.8%)
unknown,107 (29.0%),13 (32.5%),1 (0.2%),98 (15.6%),219 (14.2%)
STR,10 (2.7%),0 (0.0%),198 (39.5%),0 (0.0%),208 (13.5%)
biopsy,0 (0.0%),0 (0.0%),54 (10.8%),0 (0.0%),54 (3.5%)
non_GTR,0 (0.0%),0 (0.0%),0 (0.0%),206 (32.7%),206 (13.4%)


In [6]:
# Optional: read NIfTI headers to check volume shape and voxel spacing.
# Slower — one header read per session x modality. Use on a filtered view.
baseline_rhuh = cohort.select(baseline_only=True, datasets=["rhuh_gbm"])
s_spatial = cohort_summary(baseline_rhuh, scan_headers=True)
s_spatial.spatial

H_mean  H_median  H_std  W_mean  W_median  W_std  D_mean  \
dataset  modality                                                             
rhuh_gbm flair     239.75     240.0   1.58  239.75     240.0   1.58  154.57   
         seg       239.75     240.0   1.58  239.75     240.0   1.58  154.57   
         t1        239.75     240.0   1.58  239.75     240.0   1.58  154.57   
         t1ce      239.75     240.0   1.58  239.75     240.0   1.58  154.57   
         t2        239.75     240.0   1.58  239.75     240.0   1.58  154.57   

                   D_median  D_std  spacing_x_mm_mean  spacing_x_mm_median  \
dataset  modality                                                            
rhuh_gbm flair        155.0   2.69                1.0                  1.0   
         seg          155.0   2.69                1.0                  1.0   
         t1           155.0   2.69                1.0                  1.0   
         t1ce         155.0   2.69                1.0                  1.0   
         t2           155.0   2.69                1.0                  1.0   

                   spacing_x_mm_std  spacing_y_mm_mean  spacing_y_mm_median  \
dataset  modality                                                             
rhuh_gbm flair                  0.0                1.0                  1.0   
         seg                    0.0                1.0                  1.0   
         t1                     0.0                1.0                  1.0   
         t1ce                   0.0                1.0                  1.0   
         t2                     0.0                1.0                  1.0   

                   spacing_y_mm_std  spacing_z_mm_mean  spacing_z_mm_median  \
dataset  modality                                                             
rhuh_gbm flair                  0.0                1.0                  1.0   
         seg                    0.0                1.0                  1.0   
         t1                     0.0                1.0                  1.0   
         t1ce                   0.0                1.0                  1.0   
         t2                     0.0                1.0                  1.0   

                   spacing_z_mm_std  n_scanned  
dataset  modality                               
rhuh_gbm flair                  0.0         40  
         seg                    0.0         40  
         t1                     0.0         40  
         t1ce                   0.0         40  
         t2                     0.0         40

## 3. Select a Study Cohort

All criteria are optional and composable. The result is a `CohortView` — an immutable, ordered selection.

In [7]:
view = cohort.select(
    baseline_only=True,          # one scan per patient (preop)
    require_complete=True,       # all four structural modalities present
    filters={
        "eor":    "GTR",         # gross total resection only
        "has_os": True,          # survival data available
    },
    # UCSF includes lower-grade glioma — restrict to GBM (WHO grade 4)
    where=lambda r: r["dataset"] != "ucsf_pdgm" or r["who_grade"] == 4,
    resolve_duplicates="drop",   # remove UPENN when BraTS duplicate exists
)

print(f"Selected: {len(view)} sessions")
cohort_summary(view).print()

Selected: 488 sessions
════════════════════════════════════════════════════════════════════════
  COHORT SUMMARY   488 patients · 488 sessions · 4 datasets
════════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────
  Demographics & survival  (patient-level)
────────────────────────────────────────────────────────────────────────
           n_pat  n_ses  n_long  age_μ  age_med  age_σ  age_miss%   os_μ  os_med   os_σ  os_miss%  event%
dataset                                                                                                  
brats2020    119    119       0   62.0     63.4   12.0        0.0  445.7   374.0  343.9       0.0    99.2
rhuh_gbm      27     27       0   63.1     65.0    8.4        0.0  493.9   395.0  292.0       0.0    74.1
ucsf_pdgm    231    231       0   60.6     61.0   12.8        0.0  556.0   430.0  429.5       0.0    55.0
upenn_gbm    111    111       0   62.8     63.4   1

In [8]:
# Views can be further narrowed by chaining .select()
train_view    = view.select(partition="train")     # BraTS + RHUH + UCSF
external_view = view.select(partition="external")  # UPENN

print(f"Train: {len(train_view)}  |  External test: {len(external_view)}")

Train: 377  |  External test: 111


In [9]:
# .to_frame() gives the underlying DataFrame for inspection or export
df = view.to_frame()
df[["dataset", "patient_id", "age", "os_days", "os_class", "eor"]].head(10)

,dataset,patient_id,age,os_days,os_class,eor
0,brats2020,BraTS20_Training_001,60.463,289.0,0,GTR
1,brats2020,BraTS20_Training_002,52.263,616.0,2,GTR
2,brats2020,BraTS20_Training_003,54.301,464.0,2,GTR
3,brats2020,BraTS20_Training_004,39.068,788.0,2,GTR
4,brats2020,BraTS20_Training_005,68.493,465.0,2,GTR
5,brats2020,BraTS20_Training_006,67.126,269.0,0,GTR
6,brats2020,BraTS20_Training_007,69.912,503.0,2,GTR
7,brats2020,BraTS20_Training_009,56.419,1155.0,2,GTR
8,brats2020,BraTS20_Training_010,48.367,515.0,2,GTR
9,brats2020,BraTS20_Training_012,65.899,495.0,2,GTR


## 4. Iterate Directly (Framework-Agnostic)

Each iteration yields a `SampleSpec` — resolved absolute paths, all clinical facts, and per-sample imaging metadata.

In [10]:
spec = next(iter(view))

print("Key:              ", spec.global_session_key)
print("Dataset:          ", spec.dataset)
print("OS days / class:  ", spec.os_days, "/", spec.os_class)
print("EOR:              ", spec.eor)
print("IDH:              ", spec.idh_status)
print("Seg convention:   ", spec.seg_convention)          # 'brats_legacy' or 'rhuh'
print("Pre-normalised:   ", spec.intensity_prenormalised) # True only for RHUH
print()
print("Paths:")
for mod, path in spec.paths.items():
    print(f"  {mod:6s}  present={spec.present[mod]}  {path}")

Key:               brats2020__BraTS20_Training_001__tp0
Dataset:           brats2020
OS days / class:   289.0 / 0
EOR:               GTR
IDH:               None
Seg convention:    brats_legacy
Pre-normalised:    False

Paths:
  t1      present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t1.nii
  t1ce    present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t1ce.nii
  t2      present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t2.nii
  flair   present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_flair.nii
  seg     present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Trai

## 5. PyTorch / Nibabel Backend

Loads volumes eagerly via nibabel. Returns raw float32 numpy arrays — **no intensity normalisation applied**. Apply `foreground_zscore` yourself, respecting the `intensity_prenormalised` flag per sample.

In [11]:
from gbm_os.transforms import foreground_zscore

dataset = view.to_torch(include_seg=True)
item = dataset[0]

print("image shape:        ", item["image"].shape)          # [C, H, W, D]
print("modality_mask:      ", item["modality_mask"])        # [C]  1=present, 0=zero-filled
print("seg shape:          ", item["seg"].shape)            # [1, H, W, D], RHUH already remapped
print("seg unique labels:  ", set(item["seg"].flatten().tolist()))
print("os_days / os_class: ", item["os_days"], "/", item["os_class"])

image shape:         (4, 240, 240, 155)
modality_mask:       [1. 1. 1. 1.]
seg shape:           (1, 240, 240, 155)


seg unique labels:   {0, 1, 2, 4}
os_days / os_class:  289.0 / 0


In [12]:
# Intensity normalisation — apply per channel, skip RHUH (already z-scored at source)
import numpy as np

spec  = next(iter(view))
image = item["image"]   # [C, H, W, D]

normalised = np.stack([
    foreground_zscore(image[c], intensity_prenormalised=spec.intensity_prenormalised)
    for c in range(image.shape[0])
])

print("Normalised image shape:", normalised.shape)
print("Ch-0 foreground mean (should be ~0):",
      normalised[0][normalised[0] > 0].mean().round(4))

Normalised image shape: (4, 240, 240, 155)
Ch-0 foreground mean (should be ~0): 0.7726


## 6. MONAI Backend

Hands file paths to a `monai.data.Dataset`. You supply the transform pipeline. Add `SegRemapd` **after** `LoadImaged` to remap RHUH seg labels — `seg_convention` is already in the data dict.

Install: `pip install gbm-manifest[monai]`

In [13]:
try:
    from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd
    from gbm_os.transforms import SegRemapd

    modalities = ["t1", "t1ce", "t2", "flair"]

    transforms = Compose([
        LoadImaged(keys=modalities + ["seg"]),
        EnsureChannelFirstd(keys=modalities + ["seg"]),
        SegRemapd(seg_key="seg"),   # reads seg_convention from dict, remaps RHUH 3->4
        # add your own spatial / intensity transforms here
    ])

    monai_ds = view.to_monai(transforms=transforms, include_seg=True)
    item = monai_ds[0]

    print("t1ce shape:              ", item["t1ce"].shape)
    print("seg shape:               ", item["seg"].shape)
    print("seg_convention:          ", item["seg_convention"])
    print("intensity_prenormalised: ", item["intensity_prenormalised"])

except ImportError:
    print("MONAI not installed. Run: pip install gbm-manifest[monai]")

MONAI not installed. Run: pip install gbm-manifest[monai]


## 7. torchio Backend

Wraps each session as a `tio.Subject`. Seg is **eagerly loaded and remapped** at construction — no extra transform needed.

Install: `pip install gbm-manifest[torchio]`

In [14]:
try:
    import torchio as tio

    tio_ds = view.to_torchio(include_seg=True)
    subject = tio_ds[0]

    print("t1ce shape:", subject["t1ce"].shape)   # [1, H, W, D]
    print("seg shape: ", subject["seg"].shape)
    print("os_days:   ", subject["os_days"])

    # Add spatial transforms via tio.Compose as usual
    spatial_transforms = tio.Compose([
        tio.RescaleIntensity(out_min_max=(0, 1)),
        tio.CropOrPad((240, 240, 155)),
    ])
    print("torchio dataset ready:", len(tio_ds), "subjects")

except ImportError:
    print("torchio not installed. Run: pip install gbm-manifest[torchio]")

torchio not installed. Run: pip install gbm-manifest[torchio]


## 8. Cross-Validation Splits

Stratified group k-fold: stratified by `os_class`, grouped by `patient_id` so no patient leaks across folds.

In [15]:
folds = train_view.split(k=5, seed=42)

for i in range(folds.k):
    tr = folds.fold(i, split="train")
    va = folds.fold(i, split="val")
    print(f"Fold {i}:  train={len(tr)}  val={len(va)}")

Fold 0:  train=296  val=81
Fold 1:  train=299  val=78
Fold 2:  train=304  val=73
Fold 3:  train=304  val=73
Fold 4:  train=305  val=72


In [16]:
# Each fold returns a CohortView — plug into any backend
fold0_train_ds = folds.fold(0, split="train").to_torch()
fold0_val_ds   = folds.fold(0, split="val").to_torch()

print("Fold 0 train:", len(fold0_train_ds))
print("Fold 0 val:  ", len(fold0_val_ds))

Fold 0 train: 296
Fold 0 val:   81


## 9. Age Normalisation (Leakage-Safe)

`AgeNormalizer` fits on the training fold only and applies those stats to validation — prevents leakage.

In [17]:
from gbm_os.transforms import AgeNormalizer

fold_train = folds.fold(0, split="train")
fold_val   = folds.fold(0, split="val")

norm = AgeNormalizer()
norm.fit(fold_train)

train_ages = norm.transform(fold_train)
val_ages   = norm.transform(fold_val)    # uses train stats — no leakage

print("Train age mean (normalised, should be ~0):", train_ages.mean().round(4))
print("Train age std  (normalised, should be ~1):", train_ages.std().round(4))
print("Val   age mean (will differ from 0):      ", val_ages.mean().round(4))

Train age mean (normalised, should be ~0): 0.0
Train age std  (normalised, should be ~1): 1.0
Val   age mean (will differ from 0):       0.2083
